# מעבדה 06 — מתנדים מצומדים

במעבדה הזו תצפו במטוטלת אחת עוצרת את עצמה לחלוטין ללא שום חיכוך, תחפשו ביד את שתי הדרכים
להתחיל את הזוג שמונעות זאת, ואז תמדדו את פיצול התדירויות הזעיר שמתברר כאחראי — קודם בתזמון
חילופין איטיים, ואחר כך מתוך ספקטרום, עם אי-ודאות על כל אחד.

הניסוי הראשון נועד להפריך משהו שאתם קרוב לוודאי מאמינים בו: שאם מניעים מטוטלת אחת, האנרגיה
נשארת בעיקר שם. היא אינה נשארת שם כלל.

עבדו לפי הסדר. היכן שהמחברת מבקשת מכם לנבא, רשמו את הניבוי שלכם בתא המיועד **לפני** שתריצו
את התא הבא.

## מפרט המודל

| | |
|---|---|
| **מערכת** | שתי מסות זהות $m$, כל אחת מעוגנת בקפיץ $k$, מחוברות בקפיץ צימוד $k_c$ |
| **דינמיקה** | $M\ddot{x} + Kx = 0$, מאונטגרת ב-velocity Verlet ונפתרת בדיוק על ידי שני אופני התנודה שלה |
| **תנאי שפה** | שני הקפיצים החיצוניים מעוגנים לתמיכות קשיחות שאינן זזות ואינן בולעות |
| **אנסמבל** | מסלול דטרמיניסטי אחד לכל מערכת תנאי התחלה; רעש המדידה גאוסי, בלתי תלוי, בעל זרע קבוע |
| **מוזנח** | ריסון והנעה — הזוג הזה חופשי ומשמר; מסות הקפיצים; אי-ליניאריות בזוויות גדולות |
| **תקף כאשר** | ההעתקים נשארים ליניאריים והצעד הרבה מתחת למחזור האופן המהיר יותר |
| **אופני כישלון** | משרעות גדולות מספיק לאי-ליניאריות שתצמיד אופנים; ריסון רשלני; קריאת תמונת האתרים כיסודית |

כל הפיזיקה חיה ב-`wavelab.coupled` — פתחו אותו וקראו אותו.


In [ ]:
# JupyterLite runs this notebook in the browser, where the course package and a few pure-
# Python libraries have to be installed into the kernel first. Under a local Jupyter they
# are already importable and this whole cell does nothing.
#
# wavelab is installed without its dependency graph on purpose: Pyodide supplies its own
# builds of numpy, scipy and matplotlib, older than the versions resolved for the
# development environment, and asking for those floors would send the installer to PyPI for
# packages that have no WebAssembly wheels. Add any new *pure-Python* dependency of
# wavelab to the list below.
try:
    import piplite
except ImportError:
    pass
else:
    await piplite.install(["pint", "ipywidgets", "jupyterquiz"])
    await piplite.install("wavelab", deps=False)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from wavelab import coupled, fourier, measurement, phasors
from wavelab.validation import scaling_exponent, seed_study

MASS = 0.5  # kg
STIFFNESS = 8.0  # N/m
COUPLING = 0.05 * STIFFNESS  # weak: the trading is slow, clean, and complete
OMEGA0 = np.sqrt(STIFFNESS / MASS)
PERIOD = 2 * np.pi / OMEGA0

MATRICES = coupled.two_mass_matrices(MASS, STIFFNESS, COUPLING)
MODES = coupled.normal_mode_solve(*MATRICES)
SPLITTING = float(np.diff(MODES.frequencies)[0])
T_EX = coupled.exchange_time(*MODES.frequencies)

# Every stochastic function takes its generator explicitly, so results are reproducible
# and no hidden global state can leak between cells.
rng = np.random.default_rng(2024)

print(f"omega0 = {OMEGA0} rad/s   period = {PERIOD:.4f} s")
print(f"mode frequencies = {MODES.frequencies}")
print(f"splitting = {SPLITTING:.5f} rad/s, which is {SPLITTING / OMEGA0:.4f} of omega0")
print(f"exchange time = {T_EX:.2f} s = {T_EX / PERIOD:.1f} oscillation periods")


## חלק 1 — הניסוי המפריך, ראשון

הניעו את מסה 1 לבדה והשאירו את מסה 2 במנוחה. הביטו בפאנל התחתון ולא בעליון: הוא מראה את
האנרגיה השייכת לכל מסה, כשהקו המקווקו מסמן את הסך הכול.

אין שום ריסון במודל הזה. מה שלא יקרה לאנרגיה של מסה 1, היא לא אבדה.


In [ ]:
dt = (2 * np.pi / MODES.frequencies[-1]) / 200
run = coupled.simulate_coupled(*MATRICES, [1.0, 0.0], [0.0, 0.0], dt, int(T_EX / dt))
energies = coupled.site_energies(*MATRICES, run.positions, run.velocities)
total = energies.sum(axis=1)
share = energies[:, 0] / total

fig, (top, bottom) = plt.subplots(2, 1, figsize=(9, 4.6), sharex=True)
top.plot(run.times, run.positions[:, 0], lw=1.2, label="mass 1")
top.plot(run.times, run.positions[:, 1], lw=1.2, label="mass 2")
top.set_ylabel("x (m)")
top.legend(fontsize=8, loc="upper right")
bottom.plot(run.times, energies[:, 0], lw=1.4, label="energy in mass 1")
bottom.plot(run.times, energies[:, 1], lw=1.4, label="energy in mass 2")
bottom.plot(run.times, total, lw=1.0, ls="--", color="0.5", label="total")
bottom.set_xlabel("t (s)")
bottom.set_ylabel("energy (J)")
bottom.legend(fontsize=8, loc="center right")
plt.tight_layout()
plt.show()

print(f"mass 1 starts with {share[0]:.1%} of the energy")
print(f"        falls to   {share.min():.2%} at t = {run.times[share.argmin()]:.1f} s")
print(f"        returns to {share[-1]:.1%} at t = {run.times[-1]:.1f} s")
print(f"the total is constant to {np.ptp(total) / total[0]:.1e} — nothing was lost")
print()
print("Mass 1 starts with 97.6% rather than 100% because the coupling spring is already")
print("stretched at t = 0, and `site_energies` shares each spring's energy between the two")
print("masses it joins. The bookkeeping is a convention; the transfer is not.")


## חלק 2 — לנבא

לפני שתריצו את התא הבא, התחייבו לתשובות.

1. אתם עומדים לנסות להתחיל את הזוג ביחסים שונים $x_2/x_1$, בשחרור ממנוחה, ולהסתכל
   ב*ספקטרום* של תנועת מסה 1. עבור יחס כללי, כמה תדירויות אתם מצפים לראות בו?
2. האם יש יחסים שעבורם תראו רק אחת? אם כן, כמה יחסים כאלה, ומהם?


**הניבוי שלכם:**

1.
2.


## חלק 3 — ציד האופנים

שחררו את הזוג ממנוחה בכמה יחסים $x_2/x_1$ וקחו את הספקטרום של העתק מסה 1. תנועה המכילה שתי
תדירויות מראה שני שיאים; תנועה בתדירות יחידה מראה אחד.

שני יחסים מתוך כל האפשריים נותנים שיא יחיד. אלה אופני התנודה העצמיים, ואתם מוצאים אותם כפי
שהייתם עושים במעבדה — בציד, לא בליכסון.

הרשומה חייבת להיות ארוכה מספיק. שתי התדירויות כאן נבדלות ב-5%, ורזולוציית הספקטרום נקבעת לפי
כמה זמן צפיתם: מחזור חילופין אחד נותן תא תדר אחד על פני הפיצול, וזה אינו מספיק כדי להפריד את
השיאים כלל. ארבעה מחזורי חילופין נוחים.


In [ ]:
DT_HUNT = 0.01
N_HUNT = int(4 * T_EX / DT_HUNT) // 2 * 2

fig, axes = plt.subplots(1, 5, figsize=(12, 2.8), sharey=True)
print("  x2/x1     peaks in the spectrum of x1")
for axis, ratio in zip(axes, (-1.0, -0.5, 0.0, 0.5, 1.0), strict=True):
    hunt = coupled.simulate_coupled(
        *MATRICES, [1.0, ratio], [0.0, 0.0], DT_HUNT, N_HUNT - 1
    )
    # `fourier.spectrum` reads its record on a grid centred on t = 0. Mirroring the trajectory
    # about its start builds that even record from a simulation that only runs forwards.
    even = np.concatenate([hunt.positions[::-1, 0], hunt.positions[1:, 0]])[:N_HUNT]
    omega, spectrum = fourier.spectrum(even, DT_HUNT)
    band = (omega > 0.8 * OMEGA0) & (omega < 1.3 * OMEGA0)
    magnitude = np.abs(spectrum[band])
    frequencies = omega[band]

    peaks = [
        i for i in range(1, len(magnitude) - 1)
        if magnitude[i] > magnitude[i - 1]
        and magnitude[i] > magnitude[i + 1]
        and magnitude[i] > 0.25 * magnitude.max()
    ]
    axis.plot(frequencies, magnitude, lw=1.1)
    axis.set_title(f"x2/x1 = {ratio:+.1f}\n{len(peaks)} peak(s)", fontsize=9)
    axis.set_xlabel("omega (rad/s)")
    print(f"  {ratio:+.2f}      {len(peaks)} peak(s) at {np.round(frequencies[peaks], 3)}")

axes[0].set_ylabel("|X|")
plt.tight_layout()
plt.show()

print()
print(f"the two single-peak ratios are the mode shapes: {np.round(MODES.shapes[:, 0], 3)}"
      f" and {np.round(MODES.shapes[:, 1], 3)}")
print(f"and their frequencies are {MODES.frequencies[0]:.4f} and {MODES.frequencies[1]:.4f} rad/s;"
      f" omega0 = {OMEGA0}")


## חלק 4 — החילופין הם הנקישה של מודול 00

הנעת מסה אחת מעוררת את שני האופנים במידה שווה, ולכן תנועת מסה 1 היא סכום שני קוסינוסים
בתדירויות קרובות — וזה בדיוק מה ש-`phasors.beat_signal` בונה. האינטגרטור מעולם לא שמע על אף
אחד מהאופנים, ולכן ההסכמה למטה היא מדידה ולא חזרה על הנאמר.


In [ ]:
beat = phasors.beat_signal(0.5, 0.5, *MODES.frequencies, run.times)
envelope = np.cos(0.5 * SPLITTING * run.times)

fig, ax = plt.subplots(figsize=(9, 3.2))
ax.plot(run.times, run.positions[:, 0], lw=1.6, color="0.75", label="simulated x1")
ax.plot(run.times, beat, lw=0.9, color="crimson", label="phasors.beat_signal")
ax.plot(run.times, envelope, lw=0.9, ls="--", color="0.5")
ax.plot(run.times, -envelope, lw=0.9, ls="--", color="0.5")
ax.set_xlabel("t (s)")
ax.set_ylabel("x1 (m)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

difference = np.max(np.abs(run.positions[:, 0] - beat))
print(f"max |simulation - beat| = {difference:.2e} m, against an amplitude of 1 m")
print(f"the envelope vanishes at t = {np.pi / SPLITTING:.2f} s = T_ex/2 = {T_EX / 2:.2f} s,")
print("which is the moment mass 1 is completely still.")


## חלק 5 — צימוד קשיח יותר, חילופין מהירים יותר

זמן החילופין הוא $2\pi/\Delta\omega$, וצימוד חלש נותן $\Delta\omega \approx \wnat k_c/k$,
ולכן $T_{\text{ex}}$ אמור לרדת כ-$1/k_c$. סרקו את הצימוד ומדדו את המעריך במקום לסמוך עליו.


In [ ]:
ratios = np.geomspace(0.005, 0.05, 10)
times, splittings = [], []
for ratio in ratios:
    matrices = coupled.two_mass_matrices(MASS, STIFFNESS, ratio * STIFFNESS)
    modes = coupled.normal_mode_solve(*matrices)
    splittings.append(float(np.diff(modes.frequencies)[0]))
    times.append(coupled.exchange_time(*modes.frequencies))

slope = scaling_exponent(ratios * STIFFNESS, times)

fig, (left, right) = plt.subplots(1, 2, figsize=(9.5, 3.2))
left.loglog(ratios, times, marker="o", ms=4, lw=1.2)
left.set_xlabel("k_c / k"), left.set_ylabel("T_ex (s)")
left.set_title(f"slope {slope:+.3f}", fontsize=9)
right.plot(ratios, np.array(splittings) / OMEGA0, marker="o", ms=4, lw=1.2, label="exact")
right.plot(ratios, ratios, lw=0.9, ls="--", color="crimson", label="estimate k_c/k")
right.set_xlabel("k_c / k"), right.set_ylabel("splitting / omega0"), right.legend(fontsize=8)
plt.tight_layout()
plt.show()

print(f"log-log slope over k_c/k in [0.005, 0.05] = {slope:+.4f}   (predicted -1)")

# Now push into strong coupling, where the estimate was never meant to work.
wide = np.geomspace(0.2, 2.0, 10)
wide_times = []
for r in wide:
    wide_matrices = coupled.two_mass_matrices(MASS, STIFFNESS, r * STIFFNESS)
    wide_modes = coupled.normal_mode_solve(*wide_matrices)
    wide_times.append(coupled.exchange_time(*wide_modes.frequencies))
print(f"log-log slope over k_c/k in [0.2, 2.0]     = "
      f"{scaling_exponent(wide * STIFFNESS, wide_times):+.4f}   <- the law is a weak-coupling law")


## חלק 6 — מדידה, עם סרגל שגיאה

שתי דרכים להוציא את הפיצול מנתונים רועשים:

- **לתזמן את החילופין.** מצאו מתי האנרגיה של מסה 1 קטנה ביותר, הכפילו, והפכו.
- **לקרוא את הספקטרום.** אתרו את שני השיאים והחסירו, באינטרפולציה בין תאים לדיוק תת-תאי.

שתיהן מודדות את אותו מספר. הן אינן מודדות אותו באותה איכות, והסיבה שווה יותר מהמספר.


In [ ]:
NOISE = 0.10  # of the initial amplitude — a realistically noisy detector
long_run = coupled.simulate_coupled(
    *MATRICES, [1.0, 0.0], [0.0, 0.0], dt, int(3 * T_EX / dt)
)


def _interpolated_peak(frequencies, magnitude, index):
    """Sub-bin peak position from three samples — the usual three-point vertex."""
    a, b, c = magnitude[index - 1], magnitude[index], magnitude[index + 1]
    return frequencies[index] + 0.5 * (a - c) / (a - 2 * b + c) * (frequencies[1] - frequencies[0])


def splitting_from_spectrum(generator):
    """Frequency domain: two peaks, subtracted."""
    noisy = measurement.add_noise(long_run.positions[:, 0], NOISE, generator)
    even = np.concatenate([noisy[::-1], noisy[1:]])
    omega, spectrum = fourier.spectrum(even[: len(even) // 2 * 2], dt)
    band = (omega > 0.9 * OMEGA0) & (omega < 1.15 * OMEGA0)
    magnitude, frequencies = np.abs(spectrum[band]), omega[band]
    peaks = [
        i for i in range(1, len(magnitude) - 1)
        if magnitude[i] > magnitude[i - 1] and magnitude[i] > magnitude[i + 1]
    ]
    peaks.sort(key=lambda i: magnitude[i], reverse=True)
    top_two = sorted(_interpolated_peak(frequencies, magnitude, i) for i in peaks[:2])
    return top_two[1] - top_two[0]


def exchange_from_timing(generator):
    """Time domain: when is mass 1 emptiest?"""
    noisy = np.column_stack(
        [measurement.add_noise(long_run.positions[:, k], NOISE, generator) for k in (0, 1)]
    )
    site = coupled.site_energies(*MATRICES, noisy, long_run.velocities)
    early = long_run.times < 1.5 * T_EX
    return 2 * long_run.times[early][np.argmin(site[early, 0])]


spectral = seed_study(splitting_from_spectrum, n_seeds=8, base_seed=5)
timed = seed_study(exchange_from_timing, n_seeds=8, base_seed=5)

print(f"splitting from the spectrum = {spectral.mean:.5f} +/- {spectral.standard_error:.5f} rad/s")
print(f"                       true = {SPLITTING:.5f} rad/s"
      f"   ({100 * abs(spectral.mean - SPLITTING) / SPLITTING:.2f}% off)")
print()
print(f"exchange time, timed directly = {timed.mean:6.2f} +/- {timed.standard_error:.2f} s")
print(f"                         true = {T_EX:6.2f} s"
      f"   ({100 * abs(timed.mean - T_EX) / T_EX:.1f}% off)")
print()
print("The two routes are not equally good, and the reason is the physics of the measurement.")
print("The spectrum averages the whole record, so noise barely moves a peak. Timing the")
print("minimum asks the data its weakest possible question — where is the signal smallest? —")
print("and noise is proportionally largest exactly there. Raise NOISE to 0.3 and re-run: the")
print("spectral answer barely moves, and the timed one falls apart completely.")


In [ ]:
# 1. The hand-solved frequencies, against the numerical solver.
assert np.isclose(MODES.frequencies[0], OMEGA0, rtol=1e-12)
assert np.isclose(MODES.frequencies[1], np.sqrt((STIFFNESS + 2 * COUPLING) / MASS), rtol=1e-12)

# 2. The symmetric mode keeps omega0 whatever the coupling — its spring never stretches.
for trial in (0.01, 1.0, 10.0):
    trial_matrices = coupled.two_mass_matrices(MASS, STIFFNESS, trial * STIFFNESS)
    assert np.isclose(
        coupled.normal_mode_solve(*trial_matrices).frequencies[0], OMEGA0, rtol=1e-12
    )

# 3. The falsifier: mass 1 empties, and the total never changes.
assert share.min() < 0.01
assert np.ptp(total) / total[0] < 1e-3

# 4. The exchange really is the beat between the two mode frequencies.
assert difference < 1e-2

# 5. The scaling law holds where it was derived.
assert abs(slope - (-1.0)) < 0.05

# 6. Both measurements find the splitting, and the spectral route finds it far better.
assert abs(spectral.mean - SPLITTING) / SPLITTING < 0.02
assert abs(timed.mean - T_EX) / T_EX < 0.5

print("all checks passed")
print(f"one pendulum emptied to {share.min():.2%} and refilled, with nothing lost")


## חלק 7 — לחקור בעצמכם

שנו את הצימוד ואת יחס ההתחלה וצפו במה שקורה לחילופין. שני דברים שווים חיפוש: היחס שעוצר את
החילופין לחלוטין, והצימוד שבו ההעברה מפסיקה להיות שלמה.


In [ ]:
import ipywidgets as widgets


def explore(coupling_ratio=0.05, start_ratio=0.0):
    matrices = coupled.two_mass_matrices(MASS, STIFFNESS, coupling_ratio * STIFFNESS)
    modes = coupled.normal_mode_solve(*matrices)
    period = coupled.exchange_time(*modes.frequencies)
    step = (2 * np.pi / modes.frequencies[-1]) / 120
    span = min(period, 60.0)
    trial = coupled.simulate_coupled(
        *matrices, [1.0, start_ratio], [0.0, 0.0], step, int(span / step)
    )
    site = coupled.site_energies(*matrices, trial.positions, trial.velocities)
    fraction = site[:, 0] / site.sum(axis=1)

    fig, (top, bottom) = plt.subplots(2, 1, figsize=(9, 4.2), sharex=True)
    top.plot(trial.times, trial.positions[:, 0], lw=1.1)
    top.plot(trial.times, trial.positions[:, 1], lw=1.1)
    top.set_ylabel("x (m)")
    bottom.plot(trial.times, site[:, 0], lw=1.3)
    bottom.plot(trial.times, site[:, 1], lw=1.3)
    bottom.plot(trial.times, site.sum(axis=1), lw=0.9, ls="--", color="0.5")
    bottom.set_xlabel("t (s)"), bottom.set_ylabel("energy (J)")
    plt.tight_layout()
    plt.show()

    print(f"splitting = {np.diff(modes.frequencies)[0]:.4f} rad/s   "
          f"T_ex = {period:.2f} s = {period / PERIOD:.1f} periods")
    print(f"mass 1's energy falls to {fraction.min():.2%} of the total over this window")


widgets.interact_manual(
    explore,
    coupling_ratio=widgets.FloatLogSlider(
        base=10, min=-2.5, max=0.5, step=0.05, value=0.05, description="k_c/k"
    ),
    start_ratio=widgets.FloatSlider(min=-1.5, max=1.5, step=0.05, value=0.0, description="x2/x1"),
);


## בחנו את הבנתכם


In [ ]:
import json
from pathlib import Path

quiz_path = Path("..") / "_quiz" / "06-coupled.json"
if quiz_path.exists():
    from jupyterquiz import display_quiz

    # Parsed here with an explicit encoding: jupyterquiz opens the file with the platform
    # default, which cannot decode the Hebrew edition of this notebook on Windows.
    display_quiz(json.loads(quiz_path.read_text(encoding="utf-8")))
else:
    print("Quiz not generated yet — run: uv run python scripts/render_quizzes.py")


## לפני שאתם הולכים

כתבו כמה משפטים על כל אחת, בתא שלמטה.

1. בחלק 1 המסה הראשונה נעצרה לחלוטין ללא שום חיכוך במודל. הסבירו היכן הייתה האנרגיה שלה
   באותו רגע, ומה בשני האופנים גרם לעצירה להיות שלמה ולא חלקית.
2. בחלק 3 מצאתם את האופנים בציד בין יחסי התחלה. זה עובד עבור שתי מסות. אמרו מה הייתם עושים
   עבור עשרים, ומדוע הציד מפסיק להיות אפשרות.
3. שתי המדידות של אותו גודל בחלק 6 נחלקו בהרבה יותר מסרגלי השגיאה שלהן. איזו הייתם מדווחים,
   ומה הייתם צריכים לומר על השנייה כדי להיות הגונים?


**התשובות שלכם:**

1.
2.
3.
